In [ ]:
import os, sys
print(sys.executable)
print(os.environ.get("DYLD_LIBRARY_PATH"))

import h5py
print(h5py.version.info)

In [ ]:
import h5py
print(h5py.version.info)

# Hello

## 1. Load Data

In [ ]:
from pathlib import Path
DATA_DOWNLOAD_DIR = Path("/Volumes") / "Crucial X9" # replace with your dataset path

In [ ]:
# Dataset Metadata
import pandas as pd

metadata_df = pd.read_csv(DATA_DOWNLOAD_DIR / "emg2pose_metadata.csv")
metadata_df.head(5)

In [ ]:
# Display users
import glob
import os

users = sorted([
    p for p in Path(DATA_DOWNLOAD_DIR, "emg_dataset/by_user").iterdir()
    if p.is_dir()
])

for i, x in enumerate(users):
    print(f"{i}: {x}")

In [ ]:
# Display sessions for chosen user

user = users[0] # choose a user

sessions = sorted(glob.glob(os.path.join(user, "*.hdf5")))
for i, x in enumerate(sessions):
    print(f"{i}: {x}")

## 2. Select Session To Evaluate On

In [ ]:
from emg2pose.data import Emg2PoseSessionData

# Select our dataset 
session = sessions[0]
print(session)
data = Emg2PoseSessionData(hdf5_path=session)

In [ ]:
# View the data
print(data.fields)
print(f"{'emg shape: ':<20} {data['emg'].shape}")
print(f"{'joint_angles shape: ':<20} {data['joint_angles'].shape}")
print(f"{'time shape: ':<20} {data['time'].shape}")

In [ ]:
# Stream EMG
from experiments.stream_emg import EmgStreamer
from IPython.display import display, update_display
disp = display("", display_id=True)

stream = EmgStreamer(data)

size = data['emg'].shape[0]

# for i in range(size):
#     disp.update(str(stream.step()))

In [ ]:
from emg2pose.utils import downsample

joint_angles_30hz = downsample(data['emg'], native_fs=2000, target_fs=30)
joint_angles_30hz.shape

In [ ]:
mask = data.no_ik_failure
mask

In [ ]:
# Show our sessions metadata

metadata_df[metadata_df["filename"] == data.metadata["filename"]]

In [ ]:
# Downscale ground-truth joint angles and display it over time

import emg2pose.visualization as visualization
from emg2pose.utils import downsample
import numpy as np

joint_angles = data["joint_angles"]
joint_angles_30hz = downsample(joint_angles, native_fs=2000, target_fs=30)

visualization.get_plotly_animation_for_joint_angles(joint_angles_30hz[0:250])

In [ ]:
# Show where inverse kinematics tracking (ground truth labeling)
visualization.ik_failure_plot(data)

## 3. Generate Predictions Of This Session
Load the EMG data in to the models

### Select Session(s) to Train On
Certain methods require training our own model using various ML methods.
We use our defined `sessions` list that are avaialbe to train on
and set start and end indeces to decide what subset of `sessions` to train on.

In [ ]:
DATA_REGIMES = ["single_session", "single_user", "multi_user", "full"]

from experiments.load_data import load_data

user_train_dict = load_data(DATA_REGIMES[0], [users[0]])
pth = list(user_train_dict.values())[0][0]
print(pth)

train_data = Emg2PoseSessionData(hdf5_path=pth)

metadata_df[metadata_df["filename"] == train_data.metadata["filename"]]


In [ ]:
from experiments.trainers import _concat_sessions

train = _concat_sessions(user_train_dict)
print(train)

### 3a. Load Checkpoint and Predict
Loads a large pretrained model from Meta Reality labs and runs it on the data

In [ ]:
from pathlib import Path
import os

checkpoint_dir = Path(DATA_DOWNLOAD_DIR) / "emg2pose_model_checkpoints"

# Download checkpoint if it does not exist
if not checkpoint_dir.exists():
    os.system(f'''
    cd {DATA_DOWNLOAD_DIR} &&
    curl "https://fb-ctrl-oss.s3.amazonaws.com/emg2pose/emg2pose_model_checkpoints.tar.gz" -o emg2pose_model_checkpoints.tar.gz &&
    tar -xvzf emg2pose_model_checkpoints.tar.gz
    ''')

In [ ]:
from emg2pose.utils import generate_hydra_config_from_overrides

config = generate_hydra_config_from_overrides(
    overrides=[
        "experiment=tracking_vemg2pose",
        f"checkpoint={DATA_DOWNLOAD_DIR}/emg2pose_model_checkpoints/regression_vemg2pose.ckpt"
    ]
)

In [ ]:
from emg2pose.lightning import Emg2PoseModule

module = Emg2PoseModule.load_from_checkpoint(
    config.checkpoint,
    network=config.network,
    optimizer=config.optimizer,
    lr_scheduler=config.lr_scheduler,
)

In [ ]:
# all data
import torch

from experiments.inference import emg2pose_inferece

preds, joint_angles, no_ik_failure = emg2pose_inferece(data, module)

print("Predictions Shape: " + str(preds.shape))
print("Ground Truth Shape " + str(joint_angles.shape))

In [ ]:
# Visualize the predictions

preds_30hz = downsample(preds, native_fs=2000, target_fs=30)
visualization.get_plotly_animation_for_joint_angles(preds_30hz[0:250], color="gray")

In [ ]:
import experiments.metrics
import importlib

importlib.reload(experiments.metrics)

from experiments.metrics import ExperimentMetrics

In [ ]:
rows = []
latency_rows = []

In [ ]:

m_emg2pose = ExperimentMetrics(preds, joint_angles, no_ik_failure)

# RAW
print(preds.shape)
print(joint_angles.shape)

m_emg2pose.plot(2000)

rows.append({
    "Test Set": "User",
    "Model": "meta_emg2pose",
    **m_emg2pose.main
})

m_emg2pose.main


In [ ]:
from experiments.stream_emg import stream_inference
from experiments.inference import emg2pose_window_inference

latency = stream_inference(data, emg2pose_window_inference, module, WINDOW=12000)

latency_rows.append({
    "Model": "emg2pose",
    **latency
})

print(latency)

### 3b Small LSTM

In [ ]:
from experiments.trainers import train_small_lstm
from experiments.load_data import concat_data

emg, joint_angles_lstm = concat_data(user_train_dict)
small_lstm_model, seq_len, ds_factor, stride = train_small_lstm(emg, joint_angles_lstm)

In [ ]:
from experiments.inference import small_lstm_inference

preds, y_gt, mask_lstm = small_lstm_inference(
                data, small_lstm_model, seq_len, ds_factor, stride
            )

In [ ]:
from experiments.metrics import ExperimentMetrics, save_experiment_table

m_lstm = ExperimentMetrics(preds, y_gt, no_ik_failure)

# RAW
print(preds.shape)
print(joint_angles.shape)

rows.append({
    "Test Set": "User",
    "Model": "lstm",
    **m_lstm.main
})

m_lstm.plot(500)

m_lstm.main


In [ ]:
from experiments.stream_emg import stream_inference
from experiments.inference import lstm_window_inference

latency = stream_inference(data, lstm_window_inference, small_lstm_model, WINDOW=seq_len, ds_factor=ds_factor)

latency_rows.append({

    "Model": "lstm",
    **latency
})
print(latency)

In [ ]:
effective_hz = 500 / stride

gt_30hz = downsample(y_gt, effective_hz, 30)
preds_30hz = downsample(preds, effective_hz, 30)

print(gt_30hz.shape)
print(preds_30hz.shape)

In [ ]:
visualization.get_plotly_animation_for_joint_angles(preds_30hz[0:250], color="gray")

In [ ]:
visualization.get_plotly_animation_for_joint_angles(gt_30hz[0:250], color="pink")

### 3c Meta Model with Limited Training Data

In [ ]:
import importlib
import experiments.trainers as tr

importlib.reload(tr)

In [ ]:
from experiments.trainers import train_emg2pose
import os
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = ""

my_emg2pose_model = train_emg2pose(user_train_dict, DATA_DOWNLOAD_DIR, epochs=5)

In [ ]:
# My EMG2Pose 
preds, joint_angles, no_ik_failure = emg2pose_inferece(data, my_emg2pose_model)

In [ ]:
# Visualize the predictions

preds_30hz = downsample(preds, native_fs=2000, target_fs=30)
visualization.get_plotly_animation_for_joint_angles(preds_30hz[0:250], color="gray")

In [ ]:
# Visualize the gt

ground_truth_30hz = downsample(joint_angles, native_fs=2000, target_fs=30)
visualization.get_plotly_animation_for_joint_angles(joint_angles_30hz[0:250], color="pink")

In [ ]:
# metrics

m_small_emg2pose = ExperimentMetrics(preds, joint_angles, no_ik_failure)

# RAW
print(preds.shape)
print(joint_angles.shape)

rows.append({
    "Test Set": "User",
    "Model": "small_emg2pose",
    **m_small_emg2pose.main
})


m_small_emg2pose.plot(500)

m_small_emg2pose.main

In [ ]:
from experiments.stream_emg import stream_inference
from experiments.inference import emg2pose_window_inference

latency = stream_inference(data, emg2pose_window_inference, my_emg2pose_model, WINDOW=12000)
latency_rows.append({
    "Model": "my_emg2pose",
    **latency
})
print(latency)

### 3d Classical Machine Learning

In [ ]:
import importlib
import experiments.metrics as metrics  # wherever your class lives

importlib.reload(metrics)

from experiments.metrics import ExperimentMetrics

In [ ]:
from experiments.trainers import build_features, train_classic_ml

emg_features, joint_angles_ml = build_features(user_train_dict)
ridge_model, svr_model, pls_model = train_classic_ml(emg_features, joint_angles_ml)

In [ ]:
from experiments.inference import classic_ml_inference

ridge_pred, svr_pred, pls_pred, gt, classic_ml_mask = classic_ml_inference(
            data, ridge_model, svr_model, pls_model
        )

In [ ]:
print("\n[Ridge]")
m_ridge = ExperimentMetrics(ridge_pred, gt, classic_ml_mask)

print(ridge_pred.shape)
print(gt.shape)
m_ridge.plot(30)
m_ridge.main

print("\n[SVR]")
m_svr = ExperimentMetrics(svr_pred, gt, classic_ml_mask)

print(svr_pred.shape)
print(gt.shape)
m_svr.plot(30)
m_svr.main


print("\n[PLS]")
m_pls = ExperimentMetrics(pls_pred, gt, classic_ml_mask)

print(pls_pred.shape)
print(gt.shape)
m_pls.plot(30)
m_pls.main

rows.append({
    "Test Set": "User",
    "Model": "ridge",
    **m_ridge.main
})

rows.append({
    "Test Set": "User",
    "Model": "svr",
    **m_svr.main
})

rows.append({
    "Test Set": "User",
    "Model": "pls",
    **m_pls.main
})



In [ ]:
from experiments.stream_emg import stream_inference
from experiments.inference import ridge_window_inference, svr_window_inference, pls_window_inference

methods = ["ridge", "svr", "pls"]
functions = [ridge_window_inference, svr_window_inference, pls_window_inference]
models = [ridge_model, svr_model, pls_model]

for i in range(len(methods)):
    latency = stream_inference(data, functions[i], models[i], WINDOW=500, STRIDE=67)
    latency_rows.append({
        "Model": methods[i],
        **latency
    })
    print(latency)




In [ ]:
# Ground Truth Visualization

visualization.get_plotly_animation_for_joint_angles(
    gt[0:250],
    color="pink"
)

In [ ]:
# Ridge Pred Visualization

visualization.get_plotly_animation_for_joint_angles(
    ridge_pred[0:250],
    color="gray"
)

In [ ]:
# SVR Pred Visualization

visualization.get_plotly_animation_for_joint_angles(
    svr_pred[0:250],
    color="gray"
)

In [ ]:
visualization.get_plotly_animation_for_joint_angles(
    pls_pred[0:250],
    color="gray"
)

In [ ]:
latency_rows